In [1]:

import os
import sys
# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, src_dir)

# Now you can import the module
import models_utils as model

# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor


from joblib import dump, load


In [2]:
transcriptomics = load("data/transcriptomics_filtered.joblib")
proteomics = load("data/proteomics_matched.joblib")

In [14]:
proteomics.shape

(369, 7372)

## Elastic Net

In [5]:
elasticnet_model = ElasticNet()

In [6]:
param_grid_elasticnet = {
    'alpha': [0.1, 0.3],            # Regularization strength
    'l1_ratio': [0.1, 0.5, 0.9],    # Mix of L1 and L2 regularizatio
    }

    #'max_iter': [1000, 5000, 10000],          # Number of iterations
    #'tol': [1e-4, 1e-3, 1e-2],                # Tolerance for optimization
    #'selection': ['cyclic', 'random'],        # Update rule for coordinate descent

In [7]:
elasticnet_scores1 = model.full_pipeline(
    model=elasticnet_model,
    param_grid=param_grid_elasticnet,
    X=transcriptomics,
    y=proteomics,
    feature_selector='anova',
    n_splits=5,
    n_samples=1,
    n_proteins=1
)


c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.287e+00, tolerance: 4.726e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.763e+00, tolerance: 4.726e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.665e+00, tolerance: 4.726e

In [8]:
param_grid_elasticnet = {
    'alpha': [0.1, 0.3],            # Regularization strength
    'l1_ratio': [0.1, 0.5, 0.9],    # Mix of L1 and L2 regularization
    'p_value': [0.1, 0.05, 0.01]    # P-values for feature selection
}

    #'max_iter': [1000, 5000, 10000],          # Number of iterations
    #'tol': [1e-4, 1e-3, 1e-2]              # Tolerance for optimization
    #'selection': ['cyclic', 'random'],        # Update rule for coordinate descent

In [9]:

elasticnet_scores = model.full_pipeline2(
    model=elasticnet_model,
    param_grid=param_grid_elasticnet,
    X=transcriptomics,
    y=proteomics,
    feature_selector='anova',
    n_splits=5,
    n_samples=1,
    n_proteins=1
)

c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.378e+01, tolerance: 4.726e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.490e+01, tolerance: 4.726e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.703e+01, tolerance: 4.726e

In [10]:
# Use the full adaptive pipeline with ElasticNet

elasticnet_scores_pe = model.full_pipeline2(
    model=elasticnet_model,
    param_grid=param_grid_elasticnet,
    X=transcriptomics,
    y=proteomics,
    feature_selector='pearson',
    n_splits=5,
    n_samples=1,
    n_proteins=1
)



c:\Users\hp\anaconda3\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.378e+01, tolerance: 4.726e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.490e+01, tolerance: 4.726e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:628:

In [11]:
elasticnet_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-0.345562,-0.350558,"{'alpha': 0.3, 'l1_ratio': 0.9, 'p_value': 0.0..."


In [12]:
elasticnet_scores_pe

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-0.345562,-0.350558,"{'alpha': 0.3, 'l1_ratio': 0.9, 'p_value': 0.0..."


In [16]:
elasticnet_scores1

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-0.345562,-0.350558,"{'alpha': 0.3, 'l1_ratio': 0.9, 'p_value': 0.0..."


## XGBoost

In [12]:
xgb = XGBRegressor(device='cpu', tree_method='hist')

In [18]:
param_grid_xgboost = {
    #'eta': [0.03],         # Learning rate (smaller values for better performance)
    'max_depth': [3, 6],            # Maximum tree depth
    #'subsample': [0.5, 0.7, 0.9, 1.0],     # Fraction of samples for training each tree
    #'colsample_bytree': [0.5, 1.0],  # Fraction of features for each tree
    #'gamma': [0, 0.1, 0.5, 1],             # Minimum loss reduction to make a further partition
    #'min_child_weight': [1, 3, 5, 7],      # Minimum sum of instance weight (Hessian) needed in a child
    #'lambda': [0, 1, 5, 10],               # L2 regularization term
    #'alpha': [0, 0.1, 0.5, 1],             # L1 regularization term
    #'n_estimators': [100, 200, 500],       # Number of boosting rounds
}

In [19]:
xgb_scores1 = model.full_pipeline(
    model = xgb,
    param_grid = param_grid_xgboost,
    X = transcriptomics,
    y = proteomics,
    feature_selector='anova',
    n_proteins = 1,
    n_samples = 1
)

In [13]:
param_grid_xgboost = {
    # 'eta': [0.03],         # Learning rate (smaller values for better performance)
    'max_depth': [3, 6],            # Maximum tree depth
    'p_value': [0.1, 0.05, 0.01],
    #'subsample': [0.5, 0.7, 0.9, 1.0],     # Fraction of samples for training each tree
    # 'colsample_bytree': [0.5, 1.0],  # Fraction of features for each tree
    #'gamma': [0, 0.1, 0.5, 1],             # Minimum loss reduction to make a further partition
    #'min_child_weight': [1, 3, 5, 7],      # Minimum sum of instance weight (Hessian) needed in a child
    #'lambda': [0, 1, 5, 10],               # L2 regularization term
    #'alpha': [0, 0.1, 0.5, 1],             # L1 regularization term
    #'n_estimators': [100, 200, 500],       # Number of boosting rounds
}

In [16]:
xgb_scores = model.full_pipeline2(
    model = xgb,
    param_grid = param_grid_xgboost,
    X = transcriptomics,
    y = proteomics,
    feature_selector='anova',
    n_proteins = 1,
    n_samples = 1
)

c:\Users\hp\anaconda3\Lib\site-packages\xgboost\core.py:160: UserWarning: [21:32:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\learner.cc:742: 
Parameters: { "max_iter" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\hp\anaconda3\Lib\site-packages\xgboost\core.py:160: UserWarning: [21:32:25] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\learner.cc:742: 
Parameters: { "max_iter" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\hp\anaconda3\Lib\site-packages\xgboost\core.py:160: UserWarning: [21:32:47] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\learner.cc:742: 
Parameters: { "max_iter" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\hp\anaconda3\Lib\site-packages\xgboost\core.py:160: UserWarning: [21:3

In [17]:
xgb_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,-0.345562,-0.527635,"{'max_depth': 6, 'p_value': 0.1, 'feature_sele..."
